In [1]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


In [2]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

In [4]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

[*********************100%***********************]  1 of 1 completed


{'missing': [], 'shape': (63, 2), 'na_total': 0}

In [5]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-AAPL_20260818-232559.csv


In [18]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population'
headers = {'User-Agent':'AFE-Homework/1.0'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

{'missing': [], 'shape': (256, 6), 'na_total': 68}

In [19]:
df_scrape.head(10)


,Location,Population,% ofworld,Date,Source (official or fromtheUnited Nations),Notes
0,World,"8,232,000,000",100%,13 Jun 2025,UN projection[1][3],
1,India,"1,429,404,000",17.3%,1 Jul 2026,Official projection[4],[b]
2,China,"1,404,890,000",17.0%,31 Dec 2025,Official estimate[5],[c]
3,United States,"341,784,857",4.1%,1 Jul 2025,Official estimate[6],[d]
4,Indonesia,"288,315,089",3.5%,31 Dec 2025,National annual projection[7],
5,Pakistan,"241,499,431",2.9%,1 Mar 2023,2023 census result[8],[e]
6,Nigeria,"223,800,000",2.7%,1 Jul 2023,Official projection[9],
7,Brazil,"213,421,037",2.6%,1 Jul 2025,Official estimate[10],
8,Bangladesh,"169,828,911",2.1%,14 Jun 2022,2022 census result[11],[f]
9,Russia,"146,028,325",1.8%,1 Jan 2025,Official estimate[13],[g]


In [20]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='countries-population')

Saved data/raw/scrape_site-wikipedia_table-countries-population_20260819-000004.csv


## Documentation
- API Source: Yahoo Finance via `yfinance` library (no Alpha Vantage key configured); symbol=AAPL, period=3mo, interval=1d, fields=date/close
- Scrape Source: https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population — table listing countries/dependencies with population figures and % of world population
- Assumptions & risks: Assumes yfinance and Wikipedia stay accessible; Wikipedia table structure may change over time (selector fragility); some rows have missing population data (na_total: 68) from smaller territories
- Confirm `.env` is not committed: Confirmed — `.env` is listed in `.gitignore` and was never committed; only `.env.example` (template with dummy values) is tracked in the repo